# TrustNet image classifier: Kaggle GPU training

Upload `TrustNet-Image-Training-Sources.zip` as a **private Kaggle dataset** and attach it here. Also attach [xhlulu/140k-real-and-fake-faces](https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces), version 2. Select a GPU accelerator and enable Internet for dependencies and pretrained weights. Run in a fresh session. Save the notebook version with outputs before the session ends.

This is a face/StyleGAN benchmark. It does not establish modern-generator, general-image, or zero subject-identity-leakage accuracy. The preparation step preserves publisher splits, removes detected byte/pixel duplicates, and groups source images; it has no verified person identities. Preserve upstream dataset licensing and attribution information. No credentials belong in this notebook or source archive.


In [ ]:
from pathlib import Path
import hashlib, json, os, sys, zipfile, subprocess, importlib.metadata
WORK = Path('/kaggle/working/trustnet-image')
WORK.mkdir(parents=True, exist_ok=True)
# Keep Kaggle's GPU torch/torchvision installation. Install other dependencies before importing them.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'numpy>=2,<3', 'scipy>=1.14,<2', 'scikit-learn>=1.6,<2', 'pillow>=10', 'PyYAML>=6,<7',
    'python-dotenv>=1,<2', 'pydantic>=2,<3', 'httpx>=0.27,<1',
    'transformers>=4.48,<6', 'huggingface-hub>=0.28,<2'])
opencv_variants = ('opencv-python', 'opencv-python-headless', 'opencv-contrib-python', 'opencv-contrib-python-headless')
installed = {}
for name in opencv_variants:
    try: installed[name] = importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError: pass
if installed != {'opencv-python': '4.14.0.94'}:
    subprocess.check_call([sys.executable, '-m', 'pip', 'uninstall', '-y', *opencv_variants])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'opencv-python==4.14.0.94'])
import torch, torchvision, cv2
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator; this notebook must not silently train on CPU.'
assert hasattr(cv2, 'CascadeClassifier') and hasattr(cv2, 'data'), 'Restart this session after installing dependencies.'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__, '| OpenCV:', cv2.__version__)


In [ ]:
packages = list(Path('/kaggle/input').rglob('TrustNet-Image-Training-Sources.zip'))
assert len(packages) == 1, 'Attach exactly one source archive dataset.'
SOURCE = WORK / 'source'
assert not SOURCE.exists(), 'Use a fresh working folder; do not overwrite a previous run.'
with zipfile.ZipFile(packages[0]) as archive:
    assert sum(item.file_size for item in archive.infolist()) < 10_000_000
    for item in archive.infolist():
        target = (SOURCE / item.filename).resolve()
        assert target.is_relative_to(SOURCE.resolve()), 'Unsafe archive path'
    archive.extractall(SOURCE)
source_manifest = json.loads((SOURCE / 'training_source_manifest.json').read_text())
for name, expected in source_manifest.items():
    assert hashlib.sha256((SOURCE / name).read_bytes()).hexdigest() == expected, name
os.chdir(SOURCE)
sys.path.insert(0, str(SOURCE))
print('Verified source files:', len(source_manifest))


## Prepare matching face crops
The CNN trains on the same Haar/eye-pair crop policy used in serving. Preparation can take substantial CPU time across 140k images. Faces that cannot be located reliably are excluded and counted. This changes the evaluated population; inspect the exclusion report. No claim of complete face coverage or subject-identity separation is made.


In [ ]:
roots = [path for path in Path('/kaggle/input').rglob('140k-real-and-fake-faces') if path.is_dir()]
assert len(roots) == 1, 'Set DATASET_ROOT to the attached xhlulu dataset folder if auto-discovery is ambiguous.'
DATASET_ROOT = roots[0]
from models.image_deepfake.training.prepare_kaggle import prepare
PREPARED = WORK / 'prepared'
provenance = prepare(DATASET_ROOT, PREPARED)
print('Retained class counts:', provenance['counts'])
print('Excluded:', len(provenance['excluded_duplicates']))
print('Grouping limitations:', provenance['grouping'])


In [ ]:
from models.image_deepfake.training.train import run_training
ARTIFACT = WORK / 'trained-artifact'
# Seed, 20-epoch maximum, validation selection and early stopping are in training.yaml.
# The untouched test split is evaluated only after selecting the best validation checkpoint.
training = run_training(str(PREPARED / 'training.yaml'), str(ARTIFACT), dry_run=False)
print('Held-out CNN metrics:', training['test_metrics'])
print('These metrics describe the retained publisher GAN-face split, not modern AI images.')


In [ ]:
from models.image_deepfake.inference.artifact import TrainedImageModel
import shutil
shutil.copy2(PREPARED / 'dataset_provenance.json', ARTIFACT / 'dataset_provenance.json')
verified = TrainedImageModel(ARTIFACT)
assert verified.ready, verified.error
print('Verified checkpoint SHA-256:', verified.manifest['weights_sha256'])
del verified
torch.cuda.empty_cache()


## Check the actual serving policy
This runs a deterministic balanced sample from the publisher test split through the complete local screening policy, including localization and abstention. It is a separate result from the CNN's full retained test metrics. Missing/occluded faces are not silently counted as correct. External model inference is disabled; downloading pinned model weights is allowed for this cell.


In [ ]:
import random
os.environ['IMAGE_MODEL_DIR'] = str(ARTIFACT)
os.environ['IMAGE_ALLOW_MODEL_DOWNLOAD'] = 'true'
os.environ['IMAGE_ALLOW_HOSTED_INFERENCE'] = 'false'
rows = json.loads((PREPARED / 'policy_test.json').read_text())
rng = random.Random(42)
sample = []
for label in (0, 1):
    candidates = [row for row in rows if row[1] == label]
    sample.extend(rng.sample(candidates, min(100, len(candidates))))
sample_path = WORK / 'serving-test-sample.json'
sample_path.write_text(json.dumps(sample))
from models.image_deepfake.evaluation.run import evaluate
policy_report = evaluate(sample_path, ARTIFACT / 'serving-policy-evaluation.json')
print('Sample size:', len(sample), '| Abstention:', policy_report['abstention_rate'])
print('Accepted decisions:', policy_report['accepted_decision_metrics'])
print('Confidence calibration is unavailable. Unseen-generator testing is still required.')


In [ ]:
MODEL_ZIP = Path('/kaggle/working/TrustNet-Image-Trained-Artifact.zip')
allowed = ['best_model.pt', 'manifest.json', 'dataset_index.json', 'training_report.json',
           'dataset_provenance.json', 'serving-policy-evaluation.json']
with zipfile.ZipFile(MODEL_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in allowed: archive.write(ARTIFACT / name, name)
print('Download:', MODEL_ZIP)
print('Archive SHA-256:', hashlib.sha256(MODEL_ZIP.read_bytes()).hexdigest())
print('Save this Kaggle notebook version with outputs. Keep training_state.pt if you need optimizer-state recovery.')


## Return to the local app
Download `TrustNet-Image-Trained-Artifact.zip` from Kaggle Output. From the local project root run:

```powershell
.\.venv\Scripts\python.exe scripts/setup/install-image-model.py --package "C:\path\TrustNet-Image-Trained-Artifact.zip"
```

The installer rejects smoke artifacts, mismatched hashes, wrong labels and incompatible preprocessing. Restart the image and scan services after installing. Verify `/ready`, rescan images, and review abstention and real-photo errors before relying on the model. This notebook does not publish a model, upload private photos, bypass GPU quotas, or establish commercial licensing rights.
